# 05 Enrichment Visualization

deeptools signal heatmaps over the GREAT observed-region-hit BED files produced in `05_functionalEnrichment.ipynb`.

## 05v.1 Initialization

In [70]:
docker_run() {
    docker run --rm -i \
        -u $(id -u):$(id -g) \
        -e MPLCONFIGDIR=/home/dalbao/.config/matplotlib \
        -v /tmp:/tmp \
        -v /home/dalbao:/home/dalbao \
        -v /etc/timezone:/etc/timezone:ro \
        -v /etc/localtime:/etc/localtime:ro \
        -w $(pwd) \
        "$@"
}

# Define software to use:
## deeptools for analysis and visualization of deep-sequencing data
deeptools() {
    docker_run quay.io/biocontainers/deeptools:3.5.6--pyhdfd78af_0 "$@"
}
deeptools plotHeatmap --version

cd /home/dalbao/AlbaoRunx3Manuscript/cutnrun
mkdir -p 05_enrichment

plotHeatmap 3.5.6


Associative array of bigWig files, same samples and order as `01_peakEDA.ipynb`.

In [71]:
declare -A bigWigFiles

for group in shCd19 shRunx3 memory early late terminal; do
    for target in Runx3 Runx1; do
        fn=${group}_${target}_log2.bigWig
        fpath="source_data/bg_corrected_bigWigs/"
        bigWigFiles[${group}_${target}]="${fpath}${fn}"
    done
done

echo Sample: ${bigWigFiles["shCd19_Runx3"]}

Sample: source_data/bg_corrected_bigWigs/shCd19_Runx3_log2.bigWig


## 05v.2 Region BED Files

Placeholder region set: the cl1/cl2 observed-region-hit BED files for the `Albao_Runx3OE` gene set
(`05_enrichment/regionHits/cl1/Albao_Runx3OE.bed`, `.../cl2/Albao_Runx3OE.bed`). cl1 is labeled
"type 1" and cl2 is labeled "type 2". Swap `tag`/`regionBeds` below to plot a different gene set.

In [72]:
tag="Albao_Runx3KD_Down"

declare -A regionBeds
regionBeds[cl1]="05_enrichment/regionHits/cl1/${tag}.bed"
regionBeds[cl2]="05_enrichment/regionHits/cl2/${tag}.bed"

declare -A regionLabels
regionLabels[cl1]="type 1"
regionLabels[cl2]="type 2"

wc -l ${regionBeds[cl1]} ${regionBeds[cl2]}

  26 05_enrichment/regionHits/cl1/Albao_Runx3KD_Down.bed
  53 05_enrichment/regionHits/cl2/Albao_Runx3KD_Down.bed
  79 total


## 05v.3 Compute Matrix

In [73]:
deeptools computeMatrix reference-point \
    -S \
    ${bigWigFiles["shCd19_Runx3"]} \
    ${bigWigFiles["shRunx3_Runx3"]} \
    ${bigWigFiles["shCd19_Runx1"]} \
    ${bigWigFiles["shRunx3_Runx1"]} \
    ${bigWigFiles["memory_Runx3"]} \
    ${bigWigFiles["early_Runx3"]} \
    ${bigWigFiles["late_Runx3"]} \
    ${bigWigFiles["terminal_Runx3"]} \
    ${bigWigFiles["memory_Runx1"]} \
    ${bigWigFiles["early_Runx1"]} \
    ${bigWigFiles["late_Runx1"]} \
    ${bigWigFiles["terminal_Runx1"]} \
    -R \
    ${regionBeds[cl1]} \
    ${regionBeds[cl2]} \
    --referencePoint center \
    -b 3000 -a 3000 \
    --numberOfProcessors 36 \
    --sortUsing mean \
    -out 05_enrichment/${tag}.gz

ls -1 05_enrichment/${tag}.gz


The following chromosome names did not match between the bigwig files
chromosome	length
              Y	  91744698
05_enrichment/Albao_Runx3KD_Down.gz


## 05v.4 Plot Heatmap

In [74]:
# deeptools plotHeatmap \
#     -m "05_enrichment/${tag}.gz" \
#     -out "05_enrichment/${tag}.pdf" \
#     --sortUsing sum \
#     --colorMap "RdYlBu_r" \
#     --regionsLabel "${regionLabels[cl1]}" "${regionLabels[cl2]}" \
#     --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
#                     "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
#                     "memory_Runx1" "early_Runx1" "late_Runx1" "terminal_Runx1"

# ls -1 05_enrichment/${tag}.pdf

## 05v.5 Plot Profile

In [76]:
deeptools plotProfile \
    -m "05_enrichment/${tag}.gz" \
    -out "05_enrichment/${tag}.profile.pdf" \
    --numPlotsPerRow 12 \
    --regionsLabel "${regionLabels[cl1]}" "${regionLabels[cl2]}" \
    --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
                    "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
                    "memory_Runx1" "early_Runx1" "late_Runx1" "terminal_Runx1" \
    --plotHeight 5 --plotWidth 5 \
    --yMax 2.40 2.40 \
        1.10 1.10 \
        1.00 1.00 1.00 1.00 \
        0.65 0.65 0.65 0.65

ls -1 05_enrichment/${tag}.profile.pdf

05_enrichment/Albao_Runx3KD_Down.profile.pdf
